In [1]:
import pandas as pd
import mysql.connector
import os

# List of CSV files and their corresponding table names
csv_files = [
    ('customers.csv', 'customers'),
    ('geolocation.csv', 'geolocation'),
    ('order_items.csv', 'order_it'),
    ('orders.csv', 'products'),
    ('payments.csv', 'delivery'),
    ('products.csv', 'payments'),
    ('sellers.csv', 'order_items') # Added payments.csv for specific handling
]

# Connect to the MySQL database
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Rbhalla@6876',
    database='project'
)
cursor = conn.cursor()

# Folder containing the CSV files
folder_path = 'C:/Users/DELL/OneDrive/Upgrad/Module Python/Python Projects/Python+SQL/Dataset'

def get_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return 'INT'
    elif pd.api.types.is_float_dtype(dtype):
        return 'FLOAT'
    elif pd.api.types.is_bool_dtype(dtype):
        return 'BOOLEAN'
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return 'DATETIME'
    else:
        return 'TEXT'

for csv_file, table_name in csv_files:
    file_path = os.path.join(folder_path, csv_file)
    
    # Read the CSV file into a pandas DataFrame
    df = pd.read_csv(file_path)
    
    # Replace NaN with None to handle SQL NULL
    df = df.where(pd.notnull(df), None)
    
    # Debugging: Check for NaN values
    print(f"Processing {csv_file}")
    print(f"NaN values before replacement:\n{df.isnull().sum()}\n")

    # Clean column names
    df.columns = [col.replace(' ', '_').replace('-', '_').replace('.', '_') for col in df.columns]

    # Generate the CREATE TABLE statement with appropriate data types
    columns = ', '.join([f'`{col}` {get_sql_type(df[col].dtype)}' for col in df.columns])
    create_table_query = f'CREATE TABLE IF NOT EXISTS `{table_name}` ({columns})'
    cursor.execute(create_table_query)

    # Insert DataFrame data into the MySQL table
    for _, row in df.iterrows():
        # Convert row to tuple and handle NaN/None explicitly
        values = tuple(None if pd.isna(x) else x for x in row)
        sql = f"INSERT INTO `{table_name}` ({', '.join(['`' + col + '`' for col in df.columns])}) VALUES ({', '.join(['%s'] * len(row))})"
        cursor.execute(sql, values)

    # Commit the transaction for the current CSV file
    conn.commit()

# Close the connection
conn.close()


Processing customers.csv
NaN values before replacement:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Processing geolocation.csv
NaN values before replacement:
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

Processing order_items.csv
NaN values before replacement:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Processing orders.csv
NaN values before replacement:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector


db = mysql.connector.connect(host = "localhost",
                            username = "root",
                            password = "Rbhalla@6876",
                            database = "brazilian_ecommerce")

cur = db.cursor()



# List all unique cities where customers are located.

In [ ]:
query = """ SELECT DISTINCT customer_city
FROM customers
ORDER BY customer_city """

cur.execute(query)

data = cur.fetchall()

# STEP 1: Convert SQL Output & Build KPI Card Visualization
# What to write:
# 1. Convert 'data' into a pandas DataFrame 'df' and preview with df.head().
# 2. Create a clean KPI summary card figure (figsize=(6, 3)) using plt.text() to display the formatted total count f"{len(df):,}" and label "Total Unique Customer Cities".
# 3. Hide axes with plt.axis("off") and call plt.show().
# Hint: df = pd.DataFrame(data) | plt.text(0.5, 0.55, f"{len(df):,}", fontsize=42, fontweight="bold", ha="center", color="#0284c7")

# Count the number of orders placed in 2017.

In [ ]:
query = """ SELECT COUNT(order_id) AS total_orders_2017
FROM orders
WHERE YEAR(order_purchase_timestamp) = 2017 """

cur.execute(query)

data = cur.fetchall()

# STEP 2: Convert Single-Row Result & Display 2017 Orders KPI Card
# What to write:
# 1. Extract total_2017 = data[0] and convert 'data' into DataFrame 'df'.
# 2. Create a KPI card figure (figsize=(6, 3)) displaying f"{df[0][0]}" in bold green (#16a34a) and subtitle "Total Orders Placed in 2017".
# 3. Turn off axes with plt.axis("off") and display via plt.show().
# Hint: df = pd.DataFrame(data) | plt.text(0.5, 0.55, f"{df[0][0]}", fontsize=42, fontweight="bold", ha="center", color="#16a34a")

# Find the total sales per category.

In [ ]:
query = """ SELECT 
    p.product_category AS category,
    ROUND(SUM(oi.price), 2) AS total_sales
FROM order_items oi
JOIN products p 
    ON oi.product_id = p.product_id
WHERE p.product_category IS NOT NULL
GROUP BY p.product_category
ORDER BY total_sales DESC """

cur.execute(query)

data = cur.fetchall()

# STEP 3: Create DataFrame & Plot Top 10 Categories Horizontal Bar Chart
# What to write:
# 1. Load 'data' into DataFrame 'df' with columns=['category', 'total_sales'], slice top_10 = df.head(10), and print(top_10).
# 2. Initialize plt.figure(figsize=(10, 6)) and plot sns.barplot(data=top_10, x="total_sales", y="category", palette="Blues_r").
# 3. Add title "Top 10 Product Categories by Total Sales", xlabel "Total Sales (BRL)", ylabel "Category", call plt.tight_layout() and plt.show().
# Hint: Horizontal bar charts prevent overlapping text labels when comparing long category names.

# Calculate the percentage of orders that were paid in installments.

In [ ]:
query = """ SELECT 
    ROUND(
        (COUNT(DISTINCT CASE WHEN payment_installments > 1 THEN order_id END) * 100.0) 
        / COUNT(DISTINCT order_id), 
    2) AS installment_order_percentage
FROM payments """

cur.execute(query)

data = cur.fetchall()

# STEP 4: Display Installment Orders Percentage KPI Card
# What to write:
# 1. Convert 'data' into a DataFrame 'df'.
# 2. Create a KPI card figure (figsize=(6, 3)) using plt.text() to show f"{df[0][0]}" and subtitle "Percentage of Installment Orders".
# 3. Call plt.axis("off") and plt.show().
# Hint: df = pd.DataFrame(data) | plt.text(0.5, 0.55, f"{df[0][0]}", fontsize=42, fontweight="bold", ha="center", color="#16a34a")

# Count the number of customers from each state. 

In [ ]:
query = """ SELECT 
    customer_state,
    COUNT(customer_id) AS total_customer_orders
FROM customers
GROUP BY customer_state """

cur.execute(query)

data = cur.fetchall()

# STEP 5: Plot Customer Distribution Across Brazilian States
# What to write:
# 1. Convert 'data' into DataFrame 'df' with columns=['State', 'Customers'].
# 2. Create plt.figure(figsize=(12, 5)) and render sns.barplot(data=df, x="State", y="Customers", color="#0284c7").
# 3. Set title "Unique Customers by Brazilian State", xlabel "State", ylabel "Unique Customers", then plt.tight_layout() and plt.show().
# Hint: Point out to students how Sao Paulo (SP) dominates customer volume compared to other states.

# Calculate the number of orders per month in 2018.

In [ ]:
query = """ SELECT 
    MONTH(order_purchase_timestamp) AS month_num,
    MONTHNAME(order_purchase_timestamp) AS month_name,
    COUNT(order_id) AS total_orders
FROM orders
WHERE YEAR(order_purchase_timestamp) = 2018
GROUP BY month_num, month_name
ORDER BY month_num """

cur.execute(query)

data = cur.fetchall()

# STEP 6: Plot 2018 Monthly Order Trend Line Chart
# What to write:
# 1. Convert 'data' into DataFrame 'df' with columns=['month_num', 'month_name', 'total_orders'].
# 2. Initialize plt.figure(figsize=(10, 5)) and plot sns.lineplot(data=df, x="month_name", y="total_orders", marker="o", linewidth=2.5, color="#0284c7").
# 3. Add title "Monthly Order Volume in 2018", rotate x-ticks by 45 degrees (plt.xticks(rotation=45)), and call plt.tight_layout() and plt.show().
# Hint: Notice the sharp drop after August 2018 because the Olist dataset ends in early September/October 2018.

# Calculate the percentage of total revenue contributed by each product category.

In [ ]:
query = """ WITH category_sales AS (
    SELECT 
        p.product_category AS category,
        SUM(oi.price) AS revenue
    FROM order_items oi
    JOIN products p 
        ON oi.product_id = p.product_id
    WHERE p.product_category IS NOT NULL
    GROUP BY p.product_category
)
SELECT 
    category,
    ROUND(revenue, 2) AS category_revenue,
    ROUND((revenue * 100.0) / SUM(revenue) OVER (), 2) AS pct_of_total_revenue
FROM category_sales
ORDER BY pct_of_total_revenue DESC """

cur.execute(query)

data = cur.fetchall()

# STEP 7: Visualize Top 10 Categories Revenue Share with a Pie Chart
# What to write:
# 1. Convert 'data' into DataFrame 'df' with columns=['category', 'category_revenue', 'pct_of_total_revenue'] and select top_10 = df.head(10).
# 2. Set title "Top 10 Categories by Revenue Contribution (%)" and plot plt.pie(top_10['pct_of_total_revenue'], labels=top_10['category'], autopct='%1.1f%%').
# 3. Display the chart using plt.show().
# Hint: plt.pie(top_10['pct_of_total_revenue'], labels=top_10['category'], autopct='%1.1f%%')

# Identify the correlation between product price and the number of times a product has been purchased.

In [ ]:
query = """  SELECT 
        product_id,
        AVG(price) AS avg_price,
        COUNT(order_item_id) AS purchase_count
    FROM order_items
    GROUP BY product_id """

cur.execute(query)
data = cur.fetchall()

# STEP 8: Compute Pearson Correlation Coefficient Using NumPy
# What to write:
# 1. Load 'data' into DataFrame 'df' with columns=["product_id", "price", "order_count"].
# 2. Extract arr1 = df["order_count"] and arr2 = df["price"].
# 3. Compute correlation matrix a = np.corrcoef([arr1, arr2]) and print("the correlation is", a[0][-1]).
# Hint: Result is ~ -0.032, showing almost zero linear correlation between price and purchase frequency (plus optional sns.scatterplot).

# Calculate the total revenue generated by each seller, and rank them by revenue.

In [ ]:
query = """  SELECT 
    s.seller_id,
    s.seller_city,
    s.seller_state,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    DENSE_RANK() OVER (ORDER BY SUM(oi.price) DESC) AS revenue_rank
FROM sellers s
JOIN order_items oi 
    ON s.seller_id = oi.seller_id
GROUP BY s.seller_id, s.seller_city, s.seller_state
ORDER BY revenue_rank
 """

cur.execute(query)
data = cur.fetchall()

# STEP 9: Plot Top 10 Ranked Sellers Horizontal Bar Chart
# What to write:
# 1. Convert 'data' into DataFrame 'df' with columns=["seller_id", "city", "state", "revenue", "rank"] and slice top_sellers = df.head(10).
# 2. Create plt.figure(figsize=(10, 5)) and plot sns.barplot(data=top_sellers, x="revenue", y="seller_id", palette="mako").
# 3. Add title "Top 10 Sellers by Total Revenue", xlabel "Total Revenue (BRL)", ylabel "Seller ID (Truncated)", then plt.tight_layout() and plt.show().
# Hint: sns.barplot(data=top_sellers, x="revenue", y="seller_id", palette="mako")

# Calculate the moving average of order values for each customer over their order history.

In [ ]:
query = """ WITH customer_orders AS (
    SELECT 
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        SUM(oi.price + oi.freight_value) AS order_value
    FROM customers c
    JOIN orders o 
        ON c.customer_id = o.customer_id
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    GROUP BY c.customer_unique_id, o.order_id, o.order_purchase_timestamp
)
SELECT 
    customer_unique_id,
    order_id,
    order_purchase_timestamp,
    ROUND(order_value, 2) AS order_value,
    ROUND(
        AVG(order_value) OVER (
            PARTITION BY customer_unique_id 
            ORDER BY order_purchase_timestamp
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 
    2) AS moving_avg_order_value
FROM customer_orders
ORDER BY customer_unique_id, order_purchase_timestamp"""

cur.execute(query)

data = cur.fetchall()

# STEP 10: Convert Moving Average Results & Visualize Customer Order Trajectory
# What to write:
# 1. Convert 'data' into DataFrame 'df' with columns=['customer_unique_id', 'order_id', 'order_purchase_timestamp', 'order_value', 'moving_avg_order_value'].
# 2. Filter for a repeat customer (or preview df.head(15)) so the line chart plots cleanly without ~96,000 overlapping timestamps.
# 3. Create plt.figure(figsize=(10, 5)), plot sns.lineplot(data=sample_df, x="order_purchase_timestamp", y="moving_avg_order_value", marker="o", linewidth=2.5, color="#0284c7"), rotate x-ticks, and call plt.show().
# Hint: Show students how ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW computes a running average per customer.

#  Calculate the cumulative sales per month for each year.

In [ ]:
query = """ WITH monthly_sales AS (
    SELECT 
        YEAR(o.order_purchase_timestamp) AS sales_year,
        MONTH(o.order_purchase_timestamp) AS sales_month,
        SUM(oi.price) AS monthly_revenue
    FROM orders o
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    GROUP BY sales_year, sales_month
)
SELECT 
    sales_year,
    sales_month,
    ROUND(monthly_revenue, 2) AS monthly_revenue,
    ROUND(
        SUM(monthly_revenue) OVER (
            PARTITION BY sales_year 
            ORDER BY sales_month
        ), 
    2) AS cumulative_yearly_sales
FROM monthly_sales
ORDER BY sales_year, sales_month"""

cur.execute(query)

data = cur.fetchall()

# STEP 11: Plot Multi-Year Cumulative Sales Growth Curves
# What to write:
# 1. Load 'data' into DataFrame 'df' with columns=['sales_year', 'sales_month', 'monthly_revenue', 'cumulative_yearly_sales'].
# 2. Initialize plt.figure(figsize=(10, 5)) and plot sns.lineplot(data=df, x="sales_month", y="cumulative_yearly_sales", hue="sales_year", marker="o", palette="tab10").
# 3. Configure title "Cumulative Monthly Sales by Year", xlabel "Month (1 = Jan, 12 = Dec)", ylabel "Cumulative Sales (BRL)", plt.xticks(range(1, 13)), plt.legend(title="Year"), plt.tight_layout(), and plt.show().
# Hint: Using hue="sales_year" overlays 2016, 2017, and 2018 trajectories on the same 12-month axis.

# Calculate the year-over-year growth rate of total sales.

In [ ]:
query = """ WITH yearly_sales AS (
    SELECT 
        YEAR(o.order_purchase_timestamp) AS sales_year,
        SUM(oi.price) AS total_sales
    FROM orders o
    JOIN order_items oi 
        ON o.order_id = oi.order_id
    GROUP BY sales_year
)
SELECT 
    sales_year,
    ROUND(total_sales, 2) AS current_year_sales,
    ROUND(LAG(total_sales) OVER (ORDER BY sales_year), 2) AS previous_year_sales,
    ROUND(
        ((total_sales - LAG(total_sales) OVER (ORDER BY sales_year)) 
        / LAG(total_sales) OVER (ORDER BY sales_year)) * 100.0, 
    2) AS yoy_growth_percentage
FROM yearly_sales
ORDER BY sales_year"""

cur.execute(query)

data = cur.fetchall()

# STEP 12: Display YoY Sales Comparison Table & Bar Chart
# What to write:
# 1. Convert 'data' into DataFrame 'df' with columns=['sales_year', 'current_year_sales', 'previous_year_sales', 'yoy_growth_percentage'] and print(df).
# 2. Initialize plt.figure(figsize=(7, 5)) and plot sns.barplot(data=df, x="sales_year", y="current_year_sales", palette="viridis").
# 3. Annotate bars or set title "Year-over-Year Total Sales & Growth (%)" and call plt.show().
# Hint: Explain why 2016 has NULL for previous_year_sales and yoy_growth_percentage due to LAG().

# Calculate the retention rate of customers, defined as the percentage of customers who make more than one purchase.

In [ ]:
# STEP 13: Write SQL Query & Visualize Customer Retention Rate
# What to write:
# 1. Write a SQL CTE 'customer_purchase_counts' joining 'customers' and 'orders' on customer_id, grouping by customer_unique_id, and counting DISTINCT order_id.
# 2. Compute retention rate percentage: ROUND((COUNT(CASE WHEN total_orders > 1 THEN 1 END) * 100.0) / COUNT(*), 2) AS retention_rate_pct.
# 3. Execute via cur.execute(query), fetch with data = cur.fetchall(), load into DataFrame 'df', and render a KPI card or donut chart showing the retention percentage (~3.12%).
# Hint: Highlight why grouping by 'customer_unique_id' (not 'customer_id') is essential in Olist because each order generates a new customer_id!